# Lesson 04: AI System Evaluation in Practice — From "It Runs" to "It's Reliable"

## Learning Objectives
- Understand the four core dimensions of AI system evaluation
- Use code to batch-test different models and systematically compare their performance
- Learn the model selection workflow: from requirements to decision
- Design a reusable evaluation guide for a real-world scenario
>
> Public benchmarks (MMLU / Chatbot Arena and friends) are covered in the concept track — see Lesson 4 of the course guide.

> Judging a single response is easy, but ensuring an entire system is reliable requires systematic evaluation methods.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys,
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [ ]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'openai'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


---

## Activity 1: "Interview" Different AI Models with Your Requirements

### Activity Goal
Design real tasks related to your work/study, test different models with the same inputs, and systematically score and compare them.
This is like "interviewing" multiple candidates to find the best fit for your needs.

In [ ]:
# Activity 1: Batch model comparison

# Define your 3 test tasks (customize based on your actual needs)
test_tasks = [
    {
        'name': 'Summarization',
        'prompt': 'Summarize the following meeting notes into 3 key points, each under 50 words:\n'
                 'Today the team discussed the Q3 product roadmap. First, the PM proposed 3 new features, '
                 'including user profile optimization, smart recommendation upgrade, and mobile adaptation. '
                 'The tech lead noted that smart recommendations would need an extra 2 weeks. '
                 'Marketing suggested prioritizing mobile adaptation since mobile users now account for 70%. '
                 'Decision: Q3 will prioritize mobile adaptation and user profile optimization; '
                 'smart recommendations deferred to Q4.'
    },
    {
        'name': 'Creative Writing',
        'prompt': 'Write a 200-word product launch invitation for "AI Smart Notebook", '
                 'targeting business professionals. Tone: professional yet enthusiastic.'
    },
    {
        'name': 'Logical Reasoning',
        'prompt': 'Xiao Ming said: "If it rains tomorrow, I won\'t go to the park." '
                 'The next day, Xiao Ming went to the park. Did it rain? Analyze the reasoning.'
    }
]

# Models to test
models = list(dict.fromkeys([MODEL, MODEL_BIG]))  # models you cannot access are skipped automatically

# Evaluation dimensions
dimensions = ['Accuracy(1-5)', 'Completeness(1-5)', 'Clarity(1-5)', 'Usefulness(1-5)']

print('Starting batch evaluation...\n')
all_results = {}

for model_name in models:
    print(f'====== Testing model: {model_name} ======')
    model_out = {}
    for task in test_tasks:
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{'role':'user','content':task['prompt']}],
                temperature=0.5)
        except Exception as e:
            # No access to this model: skip it instead of killing the comparison
            print(f'[SKIP] {model_name} unavailable: {str(e)[:100]}')
            model_out = {}
            break
        model_out[task['name']] = r.choices[0].message.content
        print(f'\n--- {task["name"]} ---')
        print(r.choices[0].message.content[:200] + '...')
    if model_out:
        all_results[model_name] = model_out

print('\nAll responses collected! Please score below.')

### Evaluation Scoring Table

In the code cell below, score each model's response for each task:

In [ ]:
# Fill in your scores based on the output above
# Format: scores[model_name][task_name] = [Accuracy, Completeness, Clarity, Usefulness]

scores = {
    MODEL: {
        'Summarization': [4, 5, 5, 4],  # adjust based on actual output
        'Creative Writing': [4, 4, 4, 3],
        'Logical Reasoning': [5, 4, 4, 4],
    },
    MODEL_BIG: {
        'Summarization': [5, 5, 5, 5],  # adjust based on actual output
        'Creative Writing': [5, 4, 5, 4],
        'Logical Reasoning': [5, 5, 5, 5],
    },
}

# Keep only the models that actually answered in the previous cell
scores = {m: s for m, s in scores.items() if m in all_results}
if not scores:
    print('No model ran in the previous cell — check your API key and PROVIDER.')

# Calculate summary scores
print(f'{"Model":<20}{"Summarization":<16}{"CreativeWriting":<16}{"LogicReasoning":<16}{"Total":<10}')
print('-' * 76)
for model, tasks in scores.items():
    row = f'{model:<20}'
    total = 0
    for task_name, sc in tasks.items():
        s = sum(sc)
        total += s
        row += f'{s:<16}'
    row += f'{total:<10}'
    print(row)

# Find the best model
if scores:
    best_model = max(scores, key=lambda m: sum(sum(v) for v in scores[m].values()))
    print(f'\nBest model: {best_model}')

### Discussion
- Do different models perform differently on different tasks?
- If you could only pick one model for daily use, which would you choose?
- Does your "scoring" match your "gut feeling"? Sometimes we feel one model is "better" but the scores say otherwise.

---

## Activity 2: Let AI Do Batch Evaluation for You

### Activity Goal
Manual evaluation above is time-consuming. Now let AI do batch evaluation — give AI a rubric and let it score automatically.

In [ ]:
# Activity 2: AI automatic batch evaluation

eval_template = (
    'Rate the following AI response (1-5 for each):\n'
    '1. Accuracy: Is the information accurate?\n'
    '2. Completeness: Does it cover all key points?\n'
    '3. Clarity: Is it clear and easy to understand?\n'
    '4. Usefulness: How helpful is it to the user?\n\n'
    'Be strict — don\'t be lenient. Output format:\n'
    'Accuracy: X/5, Completeness: X/5, Clarity: X/5, Usefulness: X/5, Total: X/20'
)

# AI auto-evaluate every task
print('AI Auto-Evaluation Results:\n')
for model_name, tasks in all_results.items():
    print(f'====== {model_name} ======')
    for task_name, answer in tasks.items():
        print(f'\n--- {task_name} ---')
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role':'system','content':'You are a strict AI evaluation expert.'},
                {'role':'user','content':f'{eval_template}\n\nTask: {task_name}\nAI Response: {answer}'}
            ],
            temperature=0.2)
        print(r.choices[0].message.content)

print('\nCompare your manual scores with AI auto-scores. How different are they?')

### Discussion
- Are the AI auto-evaluation results consistent with your manual scores?
- If you had 100 responses to evaluate, what would you do?
- What are the pros and cons of using AI as a judge?

---

## Activity 3: Design Your "Evaluation Guide"

### Activity Goal
Imagine you are building an "AI Customer Service" system. Design a complete evaluation guide. This exercise helps you understand the systematic nature of evaluation.

In [ ]:
# Activity 3: Design an evaluation guide for "AI Customer Service"

# Your 5 evaluation criteria
eval_guide = '''
AI Customer Service Evaluation Guide v1.0
==========================================

Criterion 1: Intent Understanding (1-5)
  1 = Completely misunderstands customer intent
  3 = Roughly understands, but has key deviations
  5 = Precisely understands, even captures implicit needs

Criterion 2: Information Accuracy (1-5)
  1 = Provides incorrect information
  3 = Mostly correct, some vagueness
  5 = Precise, complete, and verifiable

Criterion 3: Tone and Friendliness (1-5)
  1 = Cold and robotic
  3 = Polite but lacks warmth
  5 = Warm, natural, makes the customer feel valued

Criterion 4: Resolution Efficiency (1-5)
  1 = Did not solve the problem at all
  3 = Partially solved, requires extra steps from customer
  5 = One-time resolution, exceeds customer expectations

Criterion 5: Safety and Compliance (1-5)
  1 = Contains sensitive info leaks or inappropriate promises
  3 = Basically safe, but wording could improve
  5 = Fully compliant, proactively guides in the right direction
'''

print(eval_guide)

# Let AI critique this evaluation guide
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'Please critique this AI customer service evaluation guide, pointing out its strengths and areas for improvement, and provide 2-3 specific suggestions:\n{eval_guide}'}],
    temperature=0.5)
print('AI Feedback on the Evaluation Guide:')
print(r.choices[0].message.content)

---

## Lesson Review

| Skill | Description |
|-------|-------------|
| Batch model comparison | Systematically test multiple models with the same inputs |
| AI auto-evaluation | Use AI for batch evaluation — efficient but requires caution |
| Evaluation guide design | Design a complete evaluation framework for real-world scenarios |

### Homework
1. Apply your evaluation guide to actually evaluate 3 AI responses and see if the criteria work well
2. Visit lmarena.ai to browse the Chatbot Arena leaderboard
3. Think: if you were to design an evaluation system for an AI application, what dimensions would you start with?